# Agent Prompt & MCP Experimentation

Simple notebook for testing prompts and MCP services.

In [1]:
# Update the second cell to include MCP support
import os
import sys
from pathlib import Path
from typing import Optional

# Add the parent directory to the Python path so we can load the existing modules
sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv("../.env")

from openai import OpenAI
from agents import Agent, Runner, function_tool
from agents.mcp import MCPServerStreamableHttp
from pymongo import MongoClient
from models import AIFilters

openai_client = OpenAI()
mongo_client = MongoClient(os.getenv("MONGODB_URI"))
db = mongo_client[os.getenv("DATABASE_NAME")]

In [2]:
# Knowledge base search
@function_tool
async def search_knowledge_base(user_query: str) -> str:
    """Search knowledge base for relevant documents."""
    collection = db["document_chunks"]
    
    response = openai_client.embeddings.create(
        input=user_query, model="text-embedding-3-small"
    )
    query_embedding = response.data[0].embedding
    
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "path": "embeddings",
                "queryVector": query_embedding,
                "numCandidates": 150,
                "limit": 5,
            }
        },
        {"$project": {"_id": 0, "source_url": 1, "title": 1, "content": 1}},
    ]
    
    results = list(collection.aggregate(pipeline))
    
    if not results:
        return "No relevant documents found."
    
    return "\n\n".join([
        f"{doc.get('title')}\n{doc.get('source_url')}\n{doc.get('content')}" 
        for doc in results
    ])

In [13]:
# Update the agent prompt cell
SYSTEM_PROMPT = """You are an AI assistant specialized in helping developers learn and implement AI solutions using C# and .NET. Your expertise includes:

**Core Responsibilities:**
- Guide developers through AI/ML concepts using .NET frameworks (ML.NET, Semantic Kernel, Azure AI services)
- Translate Python AI examples and tutorials into equivalent C#/.NET code
- Provide practical, working code examples with proper error handling and security best practices
- Explain AI concepts in the context of .NET development patterns and conventions
- Create and test .NET sample code in a sandboxed environment to verify code works

**Available Tools:**
- search_knowledge_base: Search Microsoft documentation and knowledge base

**When answering questions:**
1. Always start by searching Microsoft documentation, starting with the learn.microsoft.com/*/dotnet/ai content
2. Always search the knowledge base for relevant documents, prioritizing content from microsoft.com urls
3. If providing code examples, use the sandbox tools to create and test the code
4. For project creation requests, use the sandbox to create actual working projects
5. When showing code examples, you can verify they compile using the sandbox
6. If no relevant documents are found, answer using a web search tool to find up-to-date information
7. Only answer questions based on the context provided by the above instructions
8. Answer succinctly and clearly, avoiding unnecessary complexity unless asked for advanced details
9. Provide links to relevant content using a markdown format like [link text](url)
10. Format code using the latest C# syntax and .NET best practices, show console code using top-level statements
11. Do not prioritize Azure related content unless the user asks for it.

Do not make up answers or provide information outside the context of C# and .NET AI development. If you don't know the answer, say "I don't know" or suggest searching the knowledge base or web for more information.
"""

In [14]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

# Update the create_agent function to include MCP
async def create_agent():
    """Create agent with MCP server support."""
    async with MCPServerStreamableHttp(
        name="Microsoft Learn Docs MCP Server",
        params={"url": "https://learn.microsoft.com/api/mcp"}
    ) as docsserver:
        agent = Agent(
            name="C# AI Buddy",
            instructions=SYSTEM_PROMPT,
            tools=[search_knowledge_base],
            mcp_servers=[docsserver]
        )
        return agent, docsserver

# Simplified async query function
async def ask_async(query: str):
    """Ask a question using the agent with MCP support."""
    async with MCPServerStreamableHttp(
        name="Microsoft Learn Docs MCP Server",
        params={"url": "https://learn.microsoft.com/api/mcp"}
    ) as docsserver:
        agent = Agent(
            name="C# AI Buddy",
            instructions=SYSTEM_PROMPT,
            tools=[search_knowledge_base],
            mcp_servers=[docsserver]
        )
        
        result = Runner.run_streamed(agent, query)
        response = ""
        
        async for event in result.stream_events():
            if event.type == "raw_response_event":
                if hasattr(event.data, 'delta') and event.data.delta:
                    response += event.data.delta
                    print(event.data.delta, end="", flush=True)
        
        print("\n")
        return response

# Add a simple synchronous wrapper for notebook use
async def ask(query: str):
    """Synchronous wrapper for notebook testing."""
    return await ask_async(query)

In [15]:
# Test queries
await ask("What is Microsoft.Extensions.AI?")

{"query":"Microsoft.Extensions.AI"}{"user_query":"Microsoft.Extensions.AI"}**Microsoft.Extensions.AI** is a set of foundational .NET libraries that provide unified C# abstractions and infrastructure for working with artificial intelligence (AI) services, including large language models (LLMs), embeddings, chat completion, and related middleware.

### Key Concepts and Features

- **Abstractions for AI Components**: It standardizes interfaces such as `IChatClient` (for chat/LLM interaction) and `IEmbeddingGenerator<TInput, TEmbedding>` (for embedding generation) so different AI service providers (OpenAI, Azure OpenAI, Ollama, etc.) can all be used interchangeably in .NET apps.
- **Provider-Agnostic**: By implementing these interfaces, any AI service provider can integrate smoothly, which allows for portability and easy switching between providers.
- **Middleware and Pipeline**: It supports middleware patterns (like caching, telemetry/observability via OpenTelemetry, and rate limiting) fa

'{"query":"Microsoft.Extensions.AI"}{"user_query":"Microsoft.Extensions.AI"}**Microsoft.Extensions.AI** is a set of foundational .NET libraries that provide unified C# abstractions and infrastructure for working with artificial intelligence (AI) services, including large language models (LLMs), embeddings, chat completion, and related middleware.\n\n### Key Concepts and Features\n\n- **Abstractions for AI Components**: It standardizes interfaces such as `IChatClient` (for chat/LLM interaction) and `IEmbeddingGenerator<TInput, TEmbedding>` (for embedding generation) so different AI service providers (OpenAI, Azure OpenAI, Ollama, etc.) can all be used interchangeably in .NET apps.\n- **Provider-Agnostic**: By implementing these interfaces, any AI service provider can integrate smoothly, which allows for portability and easy switching between providers.\n- **Middleware and Pipeline**: It supports middleware patterns (like caching, telemetry/observability via OpenTelemetry, and rate limit

In [9]:
await ask("Show me how to create a simple chat application")

{"user_query":"create simple chat application C# .NET"}Here's a step-by-step guide for creating a simple C# chat application using .NET 8 (or later) and the OpenAI .NET API. This example outlines a minimal web API backend for sending and receiving chat messages with an AI (such as GPT-4o), plus a reference to a ready-made, modern web app template using .NET.

---

## Option 1: Use the .NET AI WebApp Template (Recommended for Fast Prototyping)

1. **Create a new AI web application using .NET template**:
   ```bash
   dotnet new aiwebapp -n GenAINetChat
   cd GenAINetChat
   ```

2. **Add the OpenAI or MCP (Model Context Protocol) NuGet package**:
   ```bash
   dotnet add package OpenAI
   # or for MCP integrations:
   dotnet add package ModelContextProtocol --prerelease
   ```

3. **Configure your API keys (e.g., OpenAI or Hugging Face) in appsettings or environment variables**.

4. **Run the app and start chatting!**
   > This gives you a fully working AI-powered chat web app with auth

'{"user_query":"create simple chat application C# .NET"}Here\'s a step-by-step guide for creating a simple C# chat application using .NET 8 (or later) and the OpenAI .NET API. This example outlines a minimal web API backend for sending and receiving chat messages with an AI (such as GPT-4o), plus a reference to a ready-made, modern web app template using .NET.\n\n---\n\n## Option 1: Use the .NET AI WebApp Template (Recommended for Fast Prototyping)\n\n1. **Create a new AI web application using .NET template**:\n   ```bash\n   dotnet new aiwebapp -n GenAINetChat\n   cd GenAINetChat\n   ```\n\n2. **Add the OpenAI or MCP (Model Context Protocol) NuGet package**:\n   ```bash\n   dotnet add package OpenAI\n   # or for MCP integrations:\n   dotnet add package ModelContextProtocol --prerelease\n   ```\n\n3. **Configure your API keys (e.g., OpenAI or Hugging Face) in appsettings or environment variables**.\n\n4. **Run the app and start chatting!**\n   > This gives you a fully working AI-powere

In [16]:
await ask("How do I create an image generation app using .NET 9?")

{"query":"image generation .NET 9"}{"user_query":"image generation app .NET 9"}To create an image generation app using **.NET 9**, you'll typically use the latest AI abstractions from Microsoft (`Microsoft.Extensions.AI`) and/or directly integrate with services like OpenAI or Azure OpenAI for text-to-image functionality. .NET 9 supports these features natively thanks to the new standard AI interfaces.

---

## 1. Prerequisites

- .NET 9 SDK ([download](https://dotnet.microsoft.com/download/dotnet/9.0))
- A supported image generation API—most commonly, [Azure OpenAI](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/dall-e?tabs=gpt-image-1) or [OpenAI](https://platform.openai.com/docs/api-reference/images)
- Add the appropriate NuGet packages:
  - `Microsoft.Extensions.AI`
  - (For Azure) `Azure.AI.OpenAI`

---

## 2. Basic Workflow

The abstraction you typically want is `IImageGenerator` from `Microsoft.Extensions.AI`, optionally wrapping the OpenAI image generator. The

'{"query":"image generation .NET 9"}{"user_query":"image generation app .NET 9"}To create an image generation app using **.NET 9**, you\'ll typically use the latest AI abstractions from Microsoft (`Microsoft.Extensions.AI`) and/or directly integrate with services like OpenAI or Azure OpenAI for text-to-image functionality. .NET 9 supports these features natively thanks to the new standard AI interfaces.\n\n---\n\n## 1. Prerequisites\n\n- .NET 9 SDK ([download](https://dotnet.microsoft.com/download/dotnet/9.0))\n- A supported image generation API—most commonly, [Azure OpenAI](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/dall-e?tabs=gpt-image-1) or [OpenAI](https://platform.openai.com/docs/api-reference/images)\n- Add the appropriate NuGet packages:\n  - `Microsoft.Extensions.AI`\n  - (For Azure) `Azure.AI.OpenAI`\n\n---\n\n## 2. Basic Workflow\n\nThe abstraction you typically want is `IImageGenerator` from `Microsoft.Extensions.AI`, optionally wrapping the OpenAI im